# Waterfall & Univariate Validation for NB_Classic_SPL_v9.4.0

Based on Yinan's v1.3.0 notebook / the Renters v1.2 vs v1.4 version.
Current release read from the feature branch via `p.*`; benchmark (v9.3.0) hardcoded.

In [ ]:
%env ENV_FOR_DYNACONF = prod
%env DYNACONF_GIT_CHECKOUT = feature/B-2937390

In [ ]:
import ltv_helpers.non_spark_helpers as nsh
import ltv_helpers.pipeline_helpers as ph
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import cm
from pyspark.sql import functions as F

from classic_spl_ltv.config.paths import paths as p

In [ ]:
pd.options.display.max_columns = 500

CURR_LABEL, PREV_LABEL = "v9.4", "v9.3"
PREV_TAG = "v9_3"

CURR_EXP_PREFIX = "e006scl"
PREV_EXP_PREFIX = "e005scl"

LINE_COL = "drv_line"

# classic internal schema -> renters names used by the rest of the notebook
RENAME = {
    "adw_pol_id": "ply_policy_id",
    "state": "ply_pt_state_cd",
    "drv_chnl_bnd": "drv_chnl_of_bnd",
    "premium": "lifetime_premium",
    "loss": "lifetime_loss",
    "cat": "cat_loss_amt",
    "lr_balance_amt": "balance_amt",
}

# Benchmark paths — check the middle folder against the bucket before running
# (your branch writes to score/, the renters release read from ltv_calc/).
PREV_ROOT = "tmx-smsiweb/classic-specialty-ltv/prod/NB_Classic_SPL_v9.3.0"
PREV_SCORES = f"{PREV_ROOT}/ltv_calc/score_internal_results/"
PREV_BALANCE = f"{PREV_ROOT}/balance/balance_factors.parquet"

In [ ]:
from ltv_helpers.spark import create_spark

spark = create_spark(buckets=p.buckets, app_name="expense")

In [ ]:
curr_cols = ph.read_parquet_s3(spark, p.score_internal_results).columns
prev_cols = ph.read_parquet_s3(spark, PREV_SCORES).columns
print(len(curr_cols), len(prev_cols))
print(sorted(prev_cols))

## I. Load Scores and merge data

In [ ]:
# Classic internal schema. Both releases should look the same apart from the
# expense prefix — check the printed prev_cols above before trusting that.
def score_cols(prefix):
    return [
        "adw_pol_id",
        "drv_line",
        "release_day",
        "state",
        "drv_chnl_bnd",
        "ple",
        f"ltv_num_{prefix}",        # -> ltv (3_deliver renames ltv_num -> ltv)
        f"ltv_den_x_mkt_{prefix}",  # -> cac_x_mkt
        f"aac_mkt_{prefix}",
        "premium",
        "loss",
        "cat",
        "lr_balance_amt",
        f"{prefix}_claims_exp",
        f"{prefix}_acquisition_exp",
        f"{prefix}_lifetime_exp",
        f"{prefix}_commission_exp_new",
        f"{prefix}_commission_exp_renew",
        "cost_of_capital",
        "cost_of_capital_20pct",
        "investment_income",
        f"{prefix}_tax",
    ]


curr_cols_to_keep = score_cols(CURR_EXP_PREFIX)
prev_cols_to_keep = score_cols(PREV_EXP_PREFIX)

missing_curr = sorted(set(curr_cols_to_keep) - set(curr_cols))
missing_prev = sorted(set(prev_cols_to_keep) - set(prev_cols))
print("missing curr:", missing_curr)
print("missing prev:", missing_prev)


def tidy(df, prefix):
    """classic names -> the renters names the rest of the notebook uses"""
    df = df.rename(columns=RENAME)
    df = df.rename(
        columns={
            f"ltv_num_{prefix}": "ltv",
            f"ltv_den_x_mkt_{prefix}": "cac_x_mkt",
            f"aac_mkt_{prefix}": "aac_mkt",
        }
    )
    df.columns = df.columns.str.replace(rf"^{prefix}_", "", regex=True)
    return df

In [ ]:
df_curr = tidy(
    nsh.read_parquet_s3_to_pandas(p.score_internal_results, columns=curr_cols_to_keep),
    CURR_EXP_PREFIX,
)
df_prev = tidy(
    nsh.read_parquet_s3_to_pandas(PREV_SCORES, columns=prev_cols_to_keep),
    PREV_EXP_PREFIX,
)
print(df_curr.shape, df_prev.shape)
print(sorted(df_curr.columns))

In [ ]:
df_prev_select = df_prev.rename(
    columns=lambda c: c if c == "ply_policy_id" else f"{c}_{PREV_TAG}"
)
df_prev_select = df_prev_select.drop(
    columns=[f"{c}_{PREV_TAG}" for c in ["drv_line", "release_day", "ply_pt_state_cd", "drv_chnl_of_bnd"]]
)

combined_df = df_curr.merge(df_prev_select, how="inner", on=["ply_policy_id"])
combined_df["release_mth_yr"] = combined_df["release_day"].astype(str).str[0:7]

print(f"curr {len(df_curr):,} | prev {len(df_prev):,} | matched {len(combined_df):,}")
print(combined_df.groupby(LINE_COL)["ply_policy_id"].count())
combined_df.head(2)

## II. Univariate Plots

In [ ]:
def plot_counts_with_two_lines(
    df,
    group_col,
    count_col,
    value_cols,  # List of two column names
    line_labels=None,  # List of two custom labels for the lines
    group_order=None,
    figsize=None,
    title=None,
    xlabel=None,
    ylabel_left="Count of Policies",
    ylabel_right=None,
    rotate_xticks=False,
):
    """
    Plots counts as bars (left y-axis) and two value columns as lines (right y-axis) grouped by group_col.
    """
    if isinstance(value_cols, str):
        value_cols = [value_cols]
    assert len(value_cols) == 2, "value_cols must be a list of two column names"

    line_styles = ["-", "-"]
    colors = ["blue", "orange"]
    markers = ["o", "o"]

    grouped = (
        df.groupby(group_col, dropna=False)
        .agg(
            count=(count_col, "count"),
            value1=(value_cols[0], "mean"),
            value2=(value_cols[1], "mean"),
        )
        .reset_index()
    )

    if figsize is None:
        figsize = (max(8, 0.28 * len(grouped)), 5)

    if group_order:
        grouped[group_col] = pd.Categorical(
            grouped[group_col], categories=group_order, ordered=True
        )
        grouped = grouped.sort_values(group_col)

    x = range(len(grouped[group_col]))

    fig, ax1 = plt.subplots(figsize=figsize)
    ax1.bar(x, grouped["count"], width=0.6, label="Count")
    ax1.set_xlabel(xlabel if xlabel else group_col)
    ax1.set_ylabel(ylabel_left)
    ax1.set_xticks(x)
    if rotate_xticks:
        ax1.set_xticklabels(grouped[group_col], rotation=90)
    else:
        ax1.set_xticklabels(grouped[group_col])

    ax2 = ax1.twinx()
    for idx, col in enumerate(["value1", "value2"]):
        label = line_labels[idx] if line_labels else f"Average {value_cols[idx]}"
        ax2.plot(
            x,
            grouped[col],
            color=colors[idx],
            marker=markers[idx],
            linestyle=line_styles[idx],
            label=label,
        )
    ax2.set_ylim(bottom=0)
    if not ylabel_right:
        ylabel_right = f"Average {', '.join(value_cols)}"
    ax2.set_ylabel(ylabel_right)

    ax1.legend(loc="upper left")
    ax2.legend(loc="upper right")

    plt.title(title or f"Count and Averages by {group_col}")
    plt.tight_layout()
    plt.show()

In [ ]:
group_dict = {
    "drv_chnl_of_bnd": "Channel of Bind",
    LINE_COL: "Line",
    "ply_pt_state_cd": "State",
    "release_mth_yr": "Release Month",
}

ROTATE = ["ply_pt_state_cd", "release_mth_yr"]

### a. LTV

In [ ]:
for group_col, group_name in group_dict.items():
    plot_counts_with_two_lines(
        df=combined_df,
        group_col=group_col,
        count_col="ply_policy_id",
        value_cols=["ltv", f"ltv_{PREV_TAG}"],
        line_labels=[f"Classic SPL LTV {CURR_LABEL}", f"Classic SPL LTV {PREV_LABEL}"],
        group_order=None,
        title=f"Classic SPL LTV: {PREV_LABEL} vs {CURR_LABEL}",
        xlabel=group_name,
        ylabel_right="Average ($)",
        rotate_xticks=(True if group_col in ROTATE else False),
    )

### b. PLE

In [ ]:
for group_col, group_name in group_dict.items():
    plot_counts_with_two_lines(
        df=combined_df,
        group_col=group_col,
        count_col="ply_policy_id",
        value_cols=["ple", f"ple_{PREV_TAG}"],
        line_labels=[f"Classic SPL PLE {CURR_LABEL}", f"Classic SPL PLE {PREV_LABEL}"],
        group_order=None,
        title=f"Classic SPL PLE: {PREV_LABEL} vs {CURR_LABEL}",
        xlabel=group_name,
        ylabel_right="PLE (years)",
        rotate_xticks=(True if group_col in ROTATE else False),
    )

### c. AAC

In [ ]:
for group_col, group_name in group_dict.items():
    plot_counts_with_two_lines(
        df=combined_df,
        group_col=group_col,
        count_col="ply_policy_id",
        value_cols=["aac_mkt", f"aac_mkt_{PREV_TAG}"],
        line_labels=[f"Classic SPL AAC {CURR_LABEL}", f"Classic SPL AAC {PREV_LABEL}"],
        group_order=None,
        title=f"Classic SPL AAC: {PREV_LABEL} vs {CURR_LABEL}",
        xlabel=group_name,
        ylabel_right="Average AAC ($)",
        rotate_xticks=(True if group_col in ROTATE else False),
    )

## III. Waterfall Plots

In [ ]:
def make_waterfall(df, components, title_input):
    components_order = list(components.keys())
    labels = [components[c][0] for c in components_order]
    signs = np.array([components[c][1] for c in components_order])

    missing = [c for c in components_order if c not in df.index]
    assert not missing, f"components not in df (would plot as 0): {missing}"
    values = np.array([df[c] for c in components_order])

    bar_heights = [values[0]]
    for i, c in enumerate(components_order[1:-1], 1):
        bar_heights.append(values[i] * signs[i])
    bar_heights.append(values[-1])

    bottoms = [0]
    cumulative = values[0]
    for h in bar_heights[1:-1]:
        bottoms.append(cumulative)
        cumulative += h
    bottoms.append(0)

    fig, ax = plt.subplots(figsize=(12, 6))
    colors = cm.tab20(np.arange(len(bar_heights)))
    ax.bar(labels, bar_heights, bottom=bottoms, color=colors)

    for i, (b, h) in enumerate(zip(bottoms, bar_heights)):
        ax.text(
            i, b + h, f"{h:.0f}", ha="center", va="bottom", fontsize=9, color="black"
        )

    ax.axhline(
        bar_heights[0] + sum(bar_heights[1:-1]),
        color="red",
        linestyle="--",
        label="Final $",
    )
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Value")
    plt.title(title_input)
    plt.legend()
    plt.tight_layout()
    plt.show()

    residual = bar_heights[0] + sum(bar_heights[1:-1]) - bar_heights[-1]
    print(
        f"change = {bar_heights[-1] - bar_heights[0]:,.2f} | unexplained = {residual:,.4f}"
    )

In [ ]:
wf_cols = [
    "lifetime_premium",
    "lifetime_loss",
    "balance_amt",
    "cat_loss_amt",
    "commission_exp_new",
    "commission_exp_renew",
    "investment_income",
    "tax",
    "ple",
    "ltv",
    "cost_of_capital",
    "cost_of_capital_20pct",
    "acquisition_exp",
    "claims_exp",
    "lifetime_exp",
    "cac_x_mkt",
    "aac_mkt",
]

for col in wf_cols:
    combined_df[f"{col}_diff"] = combined_df[f"{col}_{PREV_TAG}"] - combined_df[col]

In [ ]:
ltv_components = {
    f"ltv_{PREV_TAG}": (f"Classic SPL {PREV_LABEL} LTV", 1),
    "lifetime_premium_diff": ("Lifetime Premium Diff", -1),
    "lifetime_loss_diff": ("Lifetime Loss Diff", 1),
    "balance_amt_diff": ("Balance Diff", 1),
    "cat_loss_amt_diff": ("Cat Loss Diff", 1),
    "claims_exp_diff": ("Claims Expense Diff", 1),
    "lifetime_exp_diff": ("Lifetime Expense Diff", 1),
    "commission_exp_renew_diff": ("Commission Renew Diff", 1),
    "cost_of_capital_diff": ("Cost of Capital Diff", 1),
    "tax_diff": ("Tax Diff", 1),
    "investment_income_diff": ("Investment Income Diff", -1),
    "ltv": (f"Classic SPL {CURR_LABEL} LTV", 0),
}

aac_components = {
    f"aac_mkt_{PREV_TAG}": (f"Classic SPL {PREV_LABEL} AAC", 1),
    "lifetime_premium_diff": ("Lifetime Premium Diff", -1),
    "lifetime_loss_diff": ("Lifetime Loss Diff", 1),
    "balance_amt_diff": ("Balance Diff", 1),
    "cat_loss_amt_diff": ("Cat Loss Diff", 1),
    "claims_exp_diff": ("Claims Expense Diff", 1),
    "lifetime_exp_diff": ("Lifetime Expense Diff", 1),
    "commission_exp_renew_diff": ("Commission Renew Diff", 1),
    "cost_of_capital_20pct_diff": ("Cost of Capital (AAC, 20%) Diff", 1),
    "tax_diff": ("Tax Diff", 1),
    "investment_income_diff": ("Investment Income Diff", -1),
    "commission_exp_new_diff": ("Commission New Diff", 1),
    "acquisition_exp_diff": ("Acquisition Expense Diff", 1),
    "aac_mkt": (f"Classic SPL {CURR_LABEL} AAC", 0),
}

### 1. Overall Charts

In [ ]:
overall_compare_averages = combined_df[
    [col for col in combined_df.columns if col.startswith(tuple(wf_cols))]
].mean()

#### a. LTV

In [ ]:
make_waterfall(
    df=overall_compare_averages,
    components=ltv_components,
    title_input=f"Overall LTV Comparison of Classic SPL {CURR_LABEL} and {PREV_LABEL}",
)

#### b. AAC

In [ ]:
make_waterfall(
    df=overall_compare_averages,
    components=aac_components,
    title_input=f"Overall AAC Comparison of Classic SPL {CURR_LABEL} and {PREV_LABEL}",
)

### 2. By Line Chart

In [ ]:
line_compare_averages = (
    combined_df.groupby(LINE_COL)[
        [col for col in combined_df.columns if col.startswith(tuple(wf_cols))]
    ]
    .mean()
    .reset_index()
)

#### a. LTV Comparison

In [ ]:
for line in line_compare_averages[LINE_COL]:
    make_waterfall(
        df=line_compare_averages.loc[line_compare_averages[LINE_COL] == line].iloc[0],
        components=ltv_components,
        title_input=f"LTV Comparison of Classic SPL {CURR_LABEL} and {PREV_LABEL} -- {line}",
    )

#### b. AAC Comparison

In [ ]:
for line in line_compare_averages[LINE_COL]:
    make_waterfall(
        df=line_compare_averages.loc[line_compare_averages[LINE_COL] == line].iloc[0],
        components=aac_components,
        title_input=f"AAC Comparison of Classic SPL {CURR_LABEL} and {PREV_LABEL} -- {line}",
    )

### 3. By Channel Chart

In [ ]:
channel_compare_averages = (
    combined_df.groupby("drv_chnl_of_bnd")[
        [col for col in combined_df.columns if col.startswith(tuple(wf_cols))]
    ]
    .mean()
    .reset_index()
)

#### a. LTV Comparison

In [ ]:
for channel in channel_compare_averages.drv_chnl_of_bnd:
    make_waterfall(
        df=channel_compare_averages.loc[
            channel_compare_averages["drv_chnl_of_bnd"] == channel
        ].iloc[0],
        components=ltv_components,
        title_input=f"LTV Comparison of Classic SPL {CURR_LABEL} and {PREV_LABEL} -- {channel}",
    )

#### b. AAC Comparison

In [ ]:
for channel in channel_compare_averages.drv_chnl_of_bnd:
    make_waterfall(
        df=channel_compare_averages.loc[
            channel_compare_averages["drv_chnl_of_bnd"] == channel
        ].iloc[0],
        components=aac_components,
        title_input=f"AAC Comparison of Classic SPL {CURR_LABEL} and {PREV_LABEL} -- {channel}",
    )

## IV. Check Balance

Classic has one row per line, so this merges instead of using `[0]`.

In [ ]:
bal_feat = nsh.read_parquet_s3_to_pandas(p.balance_factors)
bal_benchmark = nsh.read_parquet_s3_to_pandas(PREV_BALANCE)
round(bal_benchmark, 3)

In [ ]:
bal_feat = round(bal_feat, 3)
bal_feat["calculated_original_lr"] = bal_feat["lr_x_cat_target"] - bal_feat["bal_factor"]
bal_feat

In [ ]:
df_agg_for_bal_pd = nsh.read_parquet_s3_to_pandas(p.agg_for_balance)

bal_check = bal_feat.merge(
    df_agg_for_bal_pd[[LINE_COL, "lr_non_cat"]], on=LINE_COL, how="outer", indicator=True
)
bal_check["lr_non_cat_rounded"] = bal_check["lr_non_cat"].round(3)
bal_check

In [ ]:
assert (bal_check["_merge"] == "both").all()
assert (bal_check["calculated_original_lr"] == bal_check["lr_non_cat_rounded"]).all()

## V. Check Expenses

In [ ]:
df_exp_2026 = nsh.read_parquet_s3_to_pandas(p.processed_expense_all)
df_exp_2026.columns = [
    col.replace(f"_{CURR_EXP_PREFIX}", "").replace("_lifetime", "_service")
    for col in df_exp_2026.columns
]
df_exp_2026.head()

In [ ]:
# classic internal keys on adw_pol_id (same name as public), so alias before joining.
# NOTE: premium_less_reinsurance is NOT in classic internal results — the acquisition /
# marketing recalc below is skipped unless you find the classic equivalent.
internal_curr = ph.read_parquet_s3(spark, p.score_internal_results).select(
    F.col("adw_pol_id").alias("int_pol_id"),
    F.col("premium_new").alias("int_premium_new"),
    F.col("premium_renew").alias("int_premium_renew"),
)
public_curr = ph.read_parquet_s3(spark, p.public_results)
public_curr = public_curr.join(
    internal_curr, on=[public_curr.adw_pol_id == internal_curr.int_pol_id]
)
public_curr.select("adw_pol_id").show(3)

In [ ]:
sel_col = [
    "line",
    "adw_pol_id",
    "release_date",
    "chnl_bnd",
    "written_premium",
    "int_premium_new",
    "int_premium_renew",
    "expense_ratio_acquisition_new",
    "expense_ratio_acquisition_renew",
    "expense_ratio_commission_new",
    "expense_ratio_commission_renew",
    "expense_ratio_service_new",
    "expense_ratio_service_renew",
    "expense_ratio_marketing_new",
    "expense_ratio_marketing_renew",
    "expense_ratio_overhead_new",
    "expense_ratio_overhead_renew",
    "expense_ratio_claims_new",
    "expense_ratio_claims_renew",
    "lifetime_exp_marketing",
    "lifetime_exp_service",
    "lifetime_exp_acquisition",
    "lifetime_exp_overhead",
    "lifetime_exp_claims",
    "lifetime_exp_commission",
    "lifetime_exp_commission_new",
    "lifetime_exp_commission_renew",
]

check_col = [
    "expense_ratio_acquisition_new",
    "expense_ratio_acquisition_renew",
    "expense_ratio_commission_new",
    "expense_ratio_commission_renew",
    "expense_ratio_service_new",
    "expense_ratio_service_renew",
    "expense_ratio_marketing_new",
    "expense_ratio_marketing_renew",
    "expense_ratio_overhead_new",
    "expense_ratio_overhead_renew",
    "expense_ratio_claims_new",
    "expense_ratio_claims_renew",
]

In [ ]:
def check_policy_exp(id):
    df = public_curr.filter((F.col("adw_pol_id") == id)).select(sel_col).toPandas()
    for item in ["service", "overhead", "claims", "commission"]:
        ratio_new_name = f"expense_ratio_{item}_new"
        ratio_renew_name = f"expense_ratio_{item}_renew"
        exp_name = f"lifetime_exp_{item}"
        exp_recalc = (
            df["int_premium_new"] * df[ratio_new_name]
            + df["int_premium_renew"] * df[ratio_renew_name]
        )
        assert df[exp_name][0] == exp_recalc[0]
    # TODO acquisition / marketing recalc needs classic's premium_less_reinsurance equivalent
    # ltv expense ratio matches processed_expense_all — classic keys on line + channel
    for col in check_col:
        ref = df_exp_2026.loc[
            (df_exp_2026["channel"] == df.chnl_bnd[0])
            & (df_exp_2026["line"] == df.line[0]),
            col,
        ]
        assert len(ref) == 1, f"{col}: expected 1 expense row, got {len(ref)}"
        assert df[col].iloc[0] == ref.iloc[0]

In [ ]:
# one policy per line, so the line -> bucket expense mapping gets exercised
sample_ids = combined_df.groupby(LINE_COL)["ply_policy_id"].first().to_dict()
sample_ids

In [ ]:
for line, pol_id in sample_ids.items():
    print(line, pol_id)
    check_policy_exp(int(pol_id))
    print("  ok")